# Requirement 7: Least-privilege application identity

> Give the calling application only the Azure permissions needed to provision, operate, and destroy the execution resources.

**Covered by:** Both

**How it is fulfilled:** The orchestrator service principal holds one role, **Container Apps SandboxGroup Data Owner**, scoped to a single sandbox group. It has no Contributor, Owner, or role-assignment rights.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
import subprocess, json
out = subprocess.run(
    ["az", "role", "assignment", "list", "--all", "--assignee", ORCH_OID, "-o", "json"],
    capture_output=True, text=True, shell=True).stdout
assignments = json.loads(out)
for a in assignments:
    print(a["roleDefinitionName"], "@", a["scope"].split("/providers/")[-1])

Container Apps SandboxGroup Data Owner @ Microsoft.App/sandboxGroups/<sandbox-group>


In [3]:
check("exactly one role assignment", len(assignments) == 1)
check("the role is SandboxGroup Data Owner", assignments[0]["roleDefinitionName"] == "Container Apps SandboxGroup Data Owner")
check("scoped to the one sandbox group, not the subscription", assignments[0]["scope"].lower() == GROUP_SCOPE.lower())

PASS  exactly one role assignment
PASS  the role is SandboxGroup Data Owner
PASS  scoped to the one sandbox group, not the subscription


In [4]:
# Negative test: the orchestrator cannot manage Azure resources or assign roles
r = requests.get(f"{ARM}/subscriptions/{SUBSCRIPTION}/resourcegroups?api-version=2021-04-01", headers=token("orchestrator", "arm"), timeout=30)
print("orchestrator lists resource groups ->", r.status_code, "| groups visible:", len(r.json().get("value", [])) if r.status_code == 200 else 0)
check("orchestrator has no control-plane reach", r.status_code in (401, 403) or not r.json().get("value"))

orchestrator lists resource groups -> 200 | groups visible: 0
PASS  orchestrator has no control-plane reach


**Caveat:** Data Owner is the only documented data-plane role, so it cannot be narrowed further today.